In [1]:
# =========================
# XAI Re-run Script (SHAP + LIME) with index-column removal
# - Uses results_{F,OF,OFR}.csv + results_OFR_trials.csv + features_used.csv
# - Selects best 1 model per set by AUPRC (tie: ECE -> Brier -> N_FEAT)
# - Removes "Unnamed: 0" (and index-like cols) from feature lists BEFORE training/XAI
# - Retrains chosen model on TRAIN, picks threshold from VAL (best F1), explains on TEST
# - Local cases: TP/FP/FN/TN each 2 cases (total up to 8)
# - Saves SHAP (summary dot/bar + waterfall per case) and LIME per case
# - Output directory structure (DAG separated):
#     ./xai_outputs/<DAG>/<SET>/
# =========================

# !pip -q install lightgbm xgboost shap lime

import os
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

import xgboost as xgb
import lightgbm as lgb

import shap
from lime.lime_tabular import LimeTabularExplainer

# -------------------------
# Paths (ALL in ./)
# -------------------------
BASE_DIR = "."

RESULTS_F_PATH       = os.path.join(BASE_DIR, "results_F.csv")
RESULTS_OF_PATH      = os.path.join(BASE_DIR, "results_OF.csv")
RESULTS_OFR_PATH     = os.path.join(BASE_DIR, "results_OFR.csv")
RESULTS_OFR_TRIALS   = os.path.join(BASE_DIR, "results_OFR_trials.csv")
FEATURES_USED_PATH   = os.path.join(BASE_DIR, "features_used.csv")

DATA_DAG = {
    "NOTEARS": {
        "train": os.path.join(BASE_DIR, "data_with_features_NOTEARS_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_NOTEARS_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_NOTEARS_test.csv"),
    },
    "PC": {
        "train": os.path.join(BASE_DIR, "data_with_features_PC_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_PC_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_PC_test.csv"),
    },
    "GES": {
        "train": os.path.join(BASE_DIR, "data_with_features_GES_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_GES_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_GES_test.csv"),
    },
    "GOLEM": {
        "train": os.path.join(BASE_DIR, "data_with_features_GOLEM_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_GOLEM_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_GOLEM_test.csv"),
    },
}

# -------------------------
# Output directory base
# -------------------------
XAI_OUT_BASE = os.path.join(BASE_DIR, "xai_outputs")
os.makedirs(XAI_OUT_BASE, exist_ok=True)

# -------------------------
# Checks
# -------------------------
for p in [RESULTS_F_PATH, RESULTS_OF_PATH, RESULTS_OFR_PATH, RESULTS_OFR_TRIALS, FEATURES_USED_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}")

for alg, paths in DATA_DAG.items():
    for k, p in paths.items():
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing {alg} {k}: {p}")

# -------------------------
# Helpers
# -------------------------
TARGET_CANDIDATES = ["label","target","y","failure","bank_failure","default","is_failed"]

def detect_target_col(df: pd.DataFrame) -> str:
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found among {TARGET_CANDIDATES}")

def normalize_k_edge(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.upper() == "ALL":
        return "ALL"
    try:
        return int(float(s))
    except Exception:
        return s

def normalize_model_name(x: str) -> str:
    s = str(x).strip()
    m = {
        "XGB": "XGBoost",
        "XGBOOST": "XGBoost",
        "LGB": "LightGBM",
        "LGBM": "LightGBM",
        "LIGHTGBM": "LightGBM",
    }
    return m.get(s.upper(), s)

def pick_best_row(df: pd.DataFrame) -> pd.Series:
    for c in ["AUPRC","ECE","Brier","N_FEAT"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in results: {c}")
    return df.sort_values(["AUPRC","ECE","Brier","N_FEAT"], ascending=[False, True, True, True]).iloc[0]

def is_index_col(c: str) -> bool:
    cl = str(c).lower().strip()
    return (
        cl.startswith("unnamed")
        or cl in {"index","_index"}
        or cl.endswith("_index")
    )

def sanitize_feature_list(cols: list[str]) -> list[str]:
    cols2 = [c for c in cols if not is_index_col(c)]
    seen = set()
    out = []
    for c in cols2:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

def load_feature_list(features_used_df: pd.DataFrame, feature_key: str) -> list[str]:
    fk = str(feature_key).strip()
    row = features_used_df.loc[features_used_df["FEATURE_KEY"] == fk]
    if row.empty:
        raise ValueError(f"FEATURE_KEY not found: {fk}")

    cols = json.loads(row["FEATURES_JSON"].iloc[0])
    cols = sanitize_feature_list(cols)
    if len(cols) == 0:
        raise ValueError(f"All features removed after dropping index-like cols. FEATURE_KEY={fk}")
    return cols

def load_split(paths: dict):
    return (
        pd.read_csv(paths["train"], low_memory=False),
        pd.read_csv(paths["val"],   low_memory=False),
        pd.read_csv(paths["test"],  low_memory=False),
    )

def make_xy(df_tr, df_va, df_te, target_col: str, feat_cols: list[str]):
    missing = [c for c in feat_cols if (c not in df_tr.columns) or (c not in df_va.columns) or (c not in df_te.columns)]
    if missing:
        raise ValueError(f"Missing features in dataset. Example: {missing[:20]} (total {len(missing)})")

    X_tr = df_tr[feat_cols].to_numpy(np.float32)
    X_va = df_va[feat_cols].to_numpy(np.float32)
    X_te = df_te[feat_cols].to_numpy(np.float32)
    y_tr = df_tr[target_col].to_numpy(np.int64)
    y_va = df_va[target_col].to_numpy(np.int64)
    y_te = df_te[target_col].to_numpy(np.int64)
    return X_tr, y_tr, X_va, y_va, X_te, y_te

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 201) -> float:
    thr_grid = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thr_grid:
        pred = (y_prob >= t).astype(int)
        tp = np.sum((pred == 1) & (y_true == 1))
        fp = np.sum((pred == 1) & (y_true == 0))
        fn = np.sum((pred == 0) & (y_true == 1))
        denom = (2 * tp + fp + fn)
        f1 = (2 * tp / denom) if denom > 0 else 0.0
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return float(best_t)

def safe_tag(s: str) -> str:
    keep = []
    for ch in str(s):
        if ch.isalnum() or ch in {"_", "-", "."}:
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

def out_dir_for(dag: str, set_name: str) -> str:
    # ./xai_outputs/<DAG>/<SET>/
    return ensure_dir(os.path.join(XAI_OUT_BASE, safe_tag(dag), safe_tag(set_name)))

# -------------------------
# Train models (XGBoost / LightGBM)
# -------------------------
RANDOM_STATE = 42

LGBM_CPU_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

def train_model(model_name: str, X_tr, y_tr, X_va, y_va):
    model_name = normalize_model_name(model_name)

    if model_name == "LightGBM":
        model = lgb.LGBMClassifier(**LGBM_CPU_PARAMS)
        model.fit(X_tr, y_tr)
        va_prob = model.predict_proba(X_va)[:, 1]
        thr = best_f1_threshold(y_va, va_prob)
        return model, thr

    if model_name == "XGBoost":
        params_gpu = dict(
            tree_method="hist",
            device="cuda",
            n_estimators=800,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        )
        params_cpu = dict(params_gpu)
        params_cpu["device"] = "cpu"

        try:
            model = xgb.XGBClassifier(**params_gpu)
            model.fit(X_tr, y_tr)
        except Exception:
            model = xgb.XGBClassifier(**params_cpu)
            model.fit(X_tr, y_tr)

        va_prob = model.predict_proba(X_va)[:, 1]
        thr = best_f1_threshold(y_va, va_prob)
        return model, thr

    raise ValueError(f"Unsupported model for XAI: {model_name}")

# -------------------------
# XAI: SHAP + LIME (PNG)
# -------------------------
def pick_local_cases_with_labels(y_true, y_prob, thr, n_each: int = 2):
    pred = (y_prob >= thr).astype(int)
    TP = np.where((pred == 1) & (y_true == 1))[0]
    FP = np.where((pred == 1) & (y_true == 0))[0]
    FN = np.where((pred == 0) & (y_true == 1))[0]
    TN = np.where((pred == 0) & (y_true == 0))[0]

    def topk_high(idxs, score, k):
        if len(idxs) == 0:
            return []
        order = idxs[np.argsort(score[idxs])[::-1]]
        return order[:k].tolist()

    def topk_low(idxs, score, k):
        if len(idxs) == 0:
            return []
        order = idxs[np.argsort(score[idxs])]
        return order[:k].tolist()

    sel = []
    sel += [(i, "TP") for i in topk_high(TP, y_prob, n_each)]
    sel += [(i, "FP") for i in topk_high(FP, y_prob, n_each)]
    sel += [(i, "FN") for i in topk_low(FN,  y_prob, n_each)]
    sel += [(i, "TN") for i in topk_low(TN,  y_prob, n_each)]

    seen = set()
    chosen, labels = [], []
    for i, lab in sel:
        if int(i) not in seen:
            chosen.append(int(i))
            labels.append(lab)
            seen.add(int(i))
    return chosen, labels

def compute_case_meta(y_true, y_prob, thr, idx: int):
    yt = int(y_true[idx])
    p = float(y_prob[idx])
    yp = int(p >= thr)
    return yt, yp, p

def run_shap_global_png(model, X_background, X_explain, feature_names, tag: str, out_dir: str,
                        max_background=2000, max_explain=5000):
    rng = np.random.RandomState(42)

    if X_background is not None and X_background.shape[0] > max_background:
        idx = rng.choice(X_background.shape[0], size=max_background, replace=False)
        Xb = X_background[idx]
    else:
        Xb = X_background

    if X_explain.shape[0] > max_explain:
        idx = rng.choice(X_explain.shape[0], size=max_explain, replace=False)
        Xe = X_explain[idx]
    else:
        Xe = X_explain

    explainer = shap.TreeExplainer(model, data=Xb, feature_perturbation="interventional")
    shap_values = explainer.shap_values(Xe, check_additivity=False)

    # summary dot
    plt.figure()
    shap.summary_plot(shap_values, Xe, feature_names=feature_names, show=False)
    p1 = os.path.join(out_dir, f"shap_summary_{safe_tag(tag)}.png")
    plt.tight_layout()
    plt.savefig(p1, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {p1}")

    # summary bar
    plt.figure()
    shap.summary_plot(shap_values, Xe, feature_names=feature_names, plot_type="bar", show=False)
    p2 = os.path.join(out_dir, f"shap_bar_{safe_tag(tag)}.png")
    plt.tight_layout()
    plt.savefig(p2, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {p2}")

    return explainer  # (optional) return

def save_shap_waterfall_png_for_case(explainer, shap_values_full, idx_in_test: int,
                                     feature_names, tag: str, case_label: str,
                                     y_true_i: int, y_pred_i: int, y_prob_i: float, thr: float,
                                     out_dir: str):
    exp = shap.Explanation(
        values=shap_values_full[idx_in_test],
        base_values=explainer.expected_value,
        data=None,
        feature_names=feature_names,
    )
    plt.figure()
    shap.plots.waterfall(exp, show=False)

    title = f"{case_label} | idx={idx_in_test} | y={y_true_i} pred={y_pred_i} p={y_prob_i:.3f} thr={thr:.3f}"
    plt.title(title, fontsize=12, pad=12)

    fname = (
        f"shap_waterfall_{safe_tag(tag)}_{case_label}"
        f"_idx{idx_in_test}_y{y_true_i}_pred{y_pred_i}_p{y_prob_i:.3f}.png"
    )
    p = os.path.join(out_dir, fname)
    plt.tight_layout()
    plt.savefig(p, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {p}")

def run_lime_png_for_cases(model, X_train, X_test, feature_names,
                           chosen_indices, chosen_labels,
                           y_true, y_prob, thr,
                           tag: str, out_dir: str, num_features=15):
    expl = LimeTabularExplainer(
        training_data=X_train,
        feature_names=feature_names,
        class_names=["ok", "fail"],
        mode="classification",
        discretize_continuous=True,
        random_state=42,
    )

    for idx, lab in zip(chosen_indices, chosen_labels):
        yt, yp, p = compute_case_meta(y_true, y_prob, thr, idx)

        exp = expl.explain_instance(
            data_row=X_test[idx],
            predict_fn=model.predict_proba,
            num_features=num_features
        )
        fig = exp.as_pyplot_figure()
        fig.suptitle(f"{lab} | idx={idx} | y={yt} pred={yp} p={p:.3f} thr={thr:.3f}",
                     fontsize=12, y=1.02)

        fname = f"lime_{safe_tag(tag)}_{lab}_idx{idx}_y{yt}_pred{yp}_p{p:.3f}.png"
        outp = os.path.join(out_dir, fname)
        fig.tight_layout()
        fig.savefig(outp, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"[SAVED] {outp}")

# -------------------------
# Load result tables
# -------------------------
dfF    = pd.read_csv(RESULTS_F_PATH)
dfOF   = pd.read_csv(RESULTS_OF_PATH)
dfOFR  = pd.read_csv(RESULTS_OFR_PATH)
dfOFRt = pd.read_csv(RESULTS_OFR_TRIALS)
feat_used = pd.read_csv(FEATURES_USED_PATH)

dfOFRt = dfOFRt.copy()
dfOFRt["MODEL_NORM"] = dfOFRt["MODEL"].apply(normalize_model_name)
dfOFRt["K_EDGE_NORM"] = dfOFRt["K_EDGE"].apply(normalize_k_edge)

# -------------------------
# Select best 1 for F / OF / OFR
# -------------------------
bestF  = pick_best_row(dfF)
bestOF = pick_best_row(dfOF)

bestOFR_mean = pick_best_row(dfOFR)
dag_o = str(bestOFR_mean["DAG"]).strip()
model_o = normalize_model_name(bestOFR_mean["MODEL"])
k_o = normalize_k_edge(bestOFR_mean["K_EDGE"])

cand = dfOFRt[(dfOFRt["DAG"] == dag_o) & (dfOFRt["MODEL_NORM"] == model_o) & (dfOFRt["K_EDGE_NORM"] == k_o)].copy()
if cand.empty:
    raise ValueError(f"No matching OFR trials for DAG={dag_o}, MODEL={model_o}, K_EDGE={k_o}")

bestOFR_trial = cand.sort_values(["AUPRC","ECE","Brier","N_FEAT"], ascending=[False, True, True, True]).iloc[0]

print("\n=== BEST (AUPRC -> ECE -> Brier -> N_FEAT) ===")
print("[F ]",  bestF[["DAG","MODEL","K_EDGE","N_FEAT","AUPRC","ECE","Brier","FEATURE_KEY"]].to_dict())
print("[OF]",  bestOF[["DAG","MODEL","K_EDGE","N_FEAT","AUPRC","ECE","Brier","FEATURE_KEY"]].to_dict())
print("[OFR mean]", bestOFR_mean[["DAG","MODEL","K_EDGE","N_FEAT","AUPRC","ECE","Brier"]].to_dict())
print("[OFR trial]", bestOFR_trial[["DAG","MODEL","K_EDGE","TRIAL","SEED","N_FEAT","AUPRC","ECE","Brier","FEATURE_KEY"]].to_dict())

# -------------------------
# Resolve features
# -------------------------
F_cols_raw   = load_feature_list(feat_used, bestF["FEATURE_KEY"])
OF_cols_raw  = load_feature_list(feat_used, bestOF["FEATURE_KEY"])
OFR_cols_raw = load_feature_list(feat_used, bestOFR_trial["FEATURE_KEY"])

print("\nFeature counts AFTER dropping index-like cols:")
print("F   n_feat =", len(F_cols_raw))
print("OF  n_feat =", len(OF_cols_raw))
print("OFR n_feat =", len(OFR_cols_raw))

# -------------------------
# One-run XAI (DAG/SET folder separated)
# -------------------------
def run_one_xai(set_name: str, dag: str, model_name: str, feat_cols: list[str],
                n_each_case: int = 2, lime_num_features: int = 15,
                max_background: int = 2000, max_explain: int = 5000):
    model_name = normalize_model_name(model_name)
    if model_name not in {"XGBoost", "LightGBM"}:
        raise ValueError(f"XAI script only supports XGBoost/LightGBM for now. got={model_name}")

    # output directory: ./xai_outputs/<DAG>/<SET>/
    out_dir = out_dir_for(dag, set_name)

    df_tr, df_va, df_te = load_split(DATA_DAG[dag])
    target_col = detect_target_col(df_tr)

    feat_cols = sanitize_feature_list(feat_cols)

    X_tr, y_tr, X_va, y_va, X_te, y_te = make_xy(df_tr, df_va, df_te, target_col, feat_cols)

    model, thr = train_model(model_name, X_tr, y_tr, X_va, y_va)
    te_prob = model.predict_proba(X_te)[:, 1]

    tag = f"{set_name}_{dag}_{model_name}_K{len(feat_cols)}"
    print(f"\n==== XAI TARGET ====")
    print(f"SET={set_name} DAG={dag} MODEL={model_name} n_feat={len(feat_cols)} thr(val-bestF1)={thr:.3f}")
    print(f"[OUT] {out_dir}")

    # ---- SHAP global (summary/bar) ----
    run_shap_global_png(
        model=model,
        X_background=X_tr,
        X_explain=X_te,
        feature_names=feat_cols,
        tag=tag,
        out_dir=out_dir,
        max_background=max_background,
        max_explain=max_explain
    )

    # ---- Local cases (TP/FP/FN/TN each 2) ----
    chosen, chosen_labels = pick_local_cases_with_labels(y_te, te_prob, thr, n_each=n_each_case)
    print("Chosen indices (test):", chosen)
    print("Chosen labels:", chosen_labels)

    # ---- SHAP per-case waterfall ----
    # Compute SHAP values on full test for correct indexing (background can be downsampled)
    rng = np.random.RandomState(42)
    Xb = X_tr
    if Xb.shape[0] > max_background:
        Xb = Xb[rng.choice(Xb.shape[0], size=max_background, replace=False)]

    explainer_full = shap.TreeExplainer(model, data=Xb, feature_perturbation="interventional")
    shap_values_full = explainer_full.shap_values(X_te, check_additivity=False)

    for idx, lab in zip(chosen, chosen_labels):
        yt, yp, p = compute_case_meta(y_te, te_prob, thr, idx)
        save_shap_waterfall_png_for_case(
            explainer=explainer_full,
            shap_values_full=shap_values_full,
            idx_in_test=idx,
            feature_names=feat_cols,
            tag=tag,
            case_label=lab,
            y_true_i=yt,
            y_pred_i=yp,
            y_prob_i=p,
            thr=thr,
            out_dir=out_dir
        )

    # ---- LIME per-case ----
    run_lime_png_for_cases(
        model=model,
        X_train=X_tr,
        X_test=X_te,
        feature_names=feat_cols,
        chosen_indices=chosen,
        chosen_labels=chosen_labels,
        y_true=y_te,
        y_prob=te_prob,
        thr=thr,
        tag=tag,
        out_dir=out_dir,
        num_features=lime_num_features
    )

# Run XAI for winners
run_one_xai("F",   str(bestF["DAG"]),         str(bestF["MODEL"]),         F_cols_raw,   n_each_case=2, lime_num_features=15)
run_one_xai("OF",  str(bestOF["DAG"]),        str(bestOF["MODEL"]),        OF_cols_raw,  n_each_case=2, lime_num_features=15)
run_one_xai("OFR", str(bestOFR_trial["DAG"]), str(bestOFR_trial["MODEL"]), OFR_cols_raw, n_each_case=2, lime_num_features=15)

print("\n[DONE] PNG outputs saved to:", XAI_OUT_BASE)


FileNotFoundError: Missing file: .\results_F.csv